<a href="https://colab.research.google.com/github/Imtiaj360/distribution_shift_with_missingness_calibration_evaluation_initial/blob/main/distribution_shift_with_missingness_calibration_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Clean up any partial state from earlier attempts ---
!rm -rf /content/tableshift_src

# --- Clone into a differently-named folder so it can't shadow the ---
# --- installed package (this was the root cause of the earlier    ---
# --- "unknown location" ImportError)                               ---
!git clone -q https://github.com/mlfoundations/tableshift.git /content/tableshift_src
!pip install -q -e /content/tableshift_src --no-deps

# --- Verify the install actually registered before moving on ---
import subprocess
r = subprocess.run(['pip', 'show', 'tableshift'], capture_output=True, text=True)
if r.stdout:
    print("tableshift installed OK:")
    print(r.stdout)
else:
    print("INSTALL FAILED. stderr below:")
    print(r.stderr)

# --- Real dependencies tableshift needs at import time ---
!pip install -q ray pyarrow tqdm torch torchvision torchinfo tables
!pip install -q fairlearn datasets folktables frozendict

# --- xport (NHANES SAS-file reader) fails to build from source on ---
# --- current Colab images. diabetes_readmission never touches that ---
# --- code path, so we stub it out instead of fighting its build.   ---
import os, sys
os.makedirs('/content/stubs/xport', exist_ok=True)
with open('/content/stubs/xport/__init__.py', 'w') as f:
    f.write("from . import v56\n")
with open('/content/stubs/xport/v56.py', 'w') as f:
    f.write(
        "def load(*a, **k):\n"
        "    raise NotImplementedError('stub xport - only needed for NHANES data')\n"
    )
sys.path.insert(0, '/content/stubs')

# --- Modeling packages ---
!pip install -q lightgbm tabpfn

print("\nCell 1 complete.")

  Preparing metadata (setup.py) ... done
tableshift installed OK:
Name: tableshift
Version: 0.1
Summary: A tabular data benchmarking toolkit.
Home-page: https://tableshift.org
Author: Josh Gardner
Author-email: jpgard@cs.washington.edu
License: 
Location: /usr/local/lib/python3.13/dist-packages
Editable project location: /content/tableshift_src
Requires: category_encoders, datasets, einops, fairlearn, folktables, frozendict, h5py, hyperopt, kaggle, numpy, openpyxl, optuna, pandas, ray, rtdl, scikit-learn, tab-transformer-pytorch, tables, torch, torchinfo, torchvision, tqdm, xport
Required-by: 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 MB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tableshift 0.1 requires category_encoders, which is not installed.
tableshift 0.1 requires fairlearn, which is not installed.
tableshift 0.1 req

In [2]:
!pip install -q <missing_package_name>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `pip install -q <missing_package_name>'


In [3]:
#2
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss
import torch

RANDOM_SEEDS = [10,11,12,13,14]        # used for the missingness mask draw
MISSING_RATE = 0.30                    # single severity level for the pilot
N_ECE_BINS = 10
DATASET_NAME = "diabetes_readmission"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. TabPFN training rows will be capped "
          "for feasibility. Colab -> Runtime -> Change runtime type -> GPU.")


Using device: cuda


In [5]:
#3
import sys
sys.path.insert(0, '/content/tableshift_src')

from tableshift import get_dataset


dset = get_dataset(DATASET_NAME)
X_train, y_train, _, _ = dset.get_pandas("train")        # source domain
X_id, y_id, _, _ = dset.get_pandas("id_test")             # in-distribution test
X_ood, y_ood, _, _ = dset.get_pandas("ood_test")           # out-of-distribution test

print(f"train={X_train.shape}  id_test={X_id.shape}  ood_test={X_ood.shape}")
print(f"positive rate: train={y_train.mean():.3f} id={y_id.mean():.3f} ood={y_ood.mean():.3f}")

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1592: UserWarning: Instantiating CategoricalDtype without any arguments.Pass a CategoricalDtype instance to silence this warning.
  df_row = df_row.select_dtypes(


train=(34288, 183)  id_test=(4287, 183)  ood_test=(50968, 183)
positive rate: train=0.424 id=0.416 ood=0.494


In [6]:
#4
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]
print(f"{len(cat_cols)} categorical cols, {len(num_cols)} numeric cols")

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if cat_cols:
    encoder.fit(X_train[cat_cols])

def encode(X):
    X = X.copy()
    if cat_cols:
        X[cat_cols] = encoder.transform(X[cat_cols])
    return X.astype(float)

X_train_enc = encode(X_train)
X_id_enc = encode(X_id)
X_ood_enc = encode(X_ood)

# Imputer fit on train only; applied AFTER missingness injection so LightGBM
# and TabPFN see identically-completed data (keeps the model comparison fair;
# the full study should also test each model's native missing-value handling).
imputer = SimpleImputer(strategy="median")
imputer.fit(X_train_enc)

def impute(X):
    return pd.DataFrame(imputer.transform(X), columns=X.columns, index=X.index)

X_train_final = impute(X_train_enc)   # training data is always complete
print("Encoding + imputer ready.")

0 categorical cols, 183 numeric cols
Encoding + imputer ready.


In [7]:
#5
def inject_mcar(X, rate, seed):
    rng = np.random.default_rng(seed)
    X = X.copy()
    mask = rng.random(X.shape) < rate
    X_arr = X.to_numpy(dtype=float)
    X_arr[mask] = np.nan
    return pd.DataFrame(X_arr, columns=X.columns, index=X.index)

print("MCAR injector ready.")

MCAR injector ready.


In [8]:
import os
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_23pjTBMYTOnZUAI5prQZDJJEIJjyRgaeIbC5HhxVdWg"

In [9]:
import os
token = os.environ.get("TABPFN_TOKEN")
print("Token is set:", bool(token))
print("Token starts with:", token[:6] + "..." if token else "N/A")

Token is set: True
Token starts with: tabpfn...


In [10]:
#6
import lightgbm as lgb
from tabpfn import TabPFNClassifier

TABPFN_CPU_ROW_CAP = 3000  # only used if no GPU

def fit_lightgbm(X, y):
    clf = lgb.LGBMClassifier(n_estimators=300, verbosity=-1, random_state=0)
    clf.fit(X, y)
    return clf

def fit_tabpfn(X, y):
    if device == "cpu" and len(X) > TABPFN_CPU_ROW_CAP:
        idx = np.random.default_rng(0).choice(len(X), TABPFN_CPU_ROW_CAP, replace=False)
        X, y = X.iloc[idx], y.iloc[idx]
        print(f"  (CPU mode: subsampled TabPFN training set to {TABPFN_CPU_ROW_CAP} rows)")
    clf = TabPFNClassifier(device=device)
    clf.fit(X, y)
    return clf

print("Fitting LightGBM...")
lgb_model = fit_lightgbm(X_train_final, y_train)

print("Fitting TabPFN...")
tabpfn_model = fit_tabpfn(X_train_final, y_train)

models = {"LightGBM": lgb_model, "TabPFN": tabpfn_model}
print("Both models fitted.")

Fitting LightGBM...
Fitting TabPFN...


tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Both models fitted.


In [11]:
#7
def expected_calibration_error(y_true, y_prob, n_bins=N_ECE_BINS):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (y_prob > lo) & (y_prob <= hi)
        if in_bin.sum() == 0:
            continue
        acc = y_true[in_bin].mean()
        conf = y_prob[in_bin].mean()
        ece += (in_bin.sum() / n) * abs(acc - conf)
    return ece

def evaluate(model, X, y):
    y_prob = model.predict_proba(X)[:, 1]
    y_prob = np.clip(y_prob, 1e-6, 1 - 1e-6)  # avoid log(0) in log loss
    return {
        "ECE": expected_calibration_error(y.to_numpy(), y_prob),
        "Brier": brier_score_loss(y, y_prob),
        "LogLoss": log_loss(y, y_prob),
    }

print("Metric functions ready.")

Metric functions ready.


In [14]:
#8
import time

results = []

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")

    t0 = time.time()
    m = evaluate(model, X_id_enc, y_id)
    print(f"  A_iid_complete done in {time.time()-t0:.1f}s")
    results.append({"model": model_name, "condition": "A_iid_complete", "seed": None, **m})

    t0 = time.time()
    m = evaluate(model, X_ood_enc, y_ood)
    print(f"  C_ood_complete done in {time.time()-t0:.1f}s")
    results.append({"model": model_name, "condition": "C_ood_complete", "seed": None, **m})

    for seed in RANDOM_SEEDS:
        t0 = time.time()
        X_miss = impute(inject_mcar(X_id_enc, MISSING_RATE, seed))
        m = evaluate(model, X_miss, y_id)
        print(f"  B_iid_missing seed={seed} done in {time.time()-t0:.1f}s")
        results.append({"model": model_name, "condition": "B_iid_missing", "seed": seed, **m})

    for seed in RANDOM_SEEDS:
        t0 = time.time()
        X_miss = impute(inject_mcar(X_ood_enc, MISSING_RATE, seed))
        m = evaluate(model, X_miss, y_ood)
        print(f"  D_ood_missing seed={seed} done in {time.time()-t0:.1f}s")
        results.append({"model": model_name, "condition": "D_ood_missing", "seed": seed, **m})

results_df = pd.DataFrame(results)
results_df.to_csv("pilot_results_raw.csv", index=False)
print(f"Done. {len(results_df)} rows collected.")

Evaluating LightGBM...
  A_iid_complete done in 0.1s
  C_ood_complete done in 0.6s
  B_iid_missing seed=10 done in 0.1s
  B_iid_missing seed=11 done in 0.1s
  B_iid_missing seed=12 done in 0.1s
  B_iid_missing seed=13 done in 0.1s
  B_iid_missing seed=14 done in 0.1s
  D_ood_missing seed=10 done in 0.9s
  D_ood_missing seed=11 done in 0.9s
  D_ood_missing seed=12 done in 0.9s
  D_ood_missing seed=13 done in 0.9s
  D_ood_missing seed=14 done in 0.9s
Evaluating TabPFN...
  A_iid_complete done in 185.8s
  C_ood_complete done in 376.3s
  B_iid_missing seed=10 done in 185.7s
  B_iid_missing seed=11 done in 185.6s
  B_iid_missing seed=12 done in 185.9s
  B_iid_missing seed=13 done in 185.9s
  B_iid_missing seed=14 done in 185.8s
  D_ood_missing seed=10 done in 376.7s
  D_ood_missing seed=11 done in 376.7s
  D_ood_missing seed=12 done in 376.9s
  D_ood_missing seed=13 done in 376.6s
  D_ood_missing seed=14 done in 377.0s
Done. 24 rows collected.


In [15]:
#9
summary = (
    results_df.groupby(["model", "condition"])[["ECE", "Brier", "LogLoss"]]
    .agg(["mean", "std"])
)
print("=== Condition summary (mean +/- std across seeds where applicable) ===")
print(summary)
summary.to_csv("pilot_results_summary.csv")

=== Condition summary (mean +/- std across seeds where applicable) ===
                              ECE               Brier             LogLoss  \
                             mean       std      mean       std      mean   
model    condition                                                          
LightGBM A_iid_complete  0.016190       NaN  0.212325       NaN  0.612231   
         B_iid_missing   0.014349  0.000771  0.221778  0.000910  0.633463   
         C_ood_complete  0.041852       NaN  0.227838       NaN  0.645468   
         D_ood_missing   0.059153  0.001166  0.236137  0.000329  0.663648   
TabPFN   A_iid_complete  0.017616       NaN  0.208997       NaN  0.603916   
         B_iid_missing   0.027180  0.006563  0.219440  0.000961  0.627772   
         C_ood_complete  0.034705       NaN  0.222006       NaN  0.631116   
         D_ood_missing   0.040693  0.000432  0.231025  0.000132  0.651289   

                                   
                              std  
model    

In [16]:
#10
print("=== Interaction check: gap = (D-A) - [(B-A)+(C-A)] = D - B - C + A ===")
print("Positive gap = evidence of a non-additive (interactive) compound failure.\n")

interaction_rows = []
for model_name in models:
    means = results_df[results_df.model == model_name].groupby("condition")[["ECE", "Brier", "LogLoss"]].mean()
    for metric in ["ECE", "Brier", "LogLoss"]:
        A = means.loc["A_iid_complete", metric]
        B = means.loc["B_iid_missing", metric]
        C = means.loc["C_ood_complete", metric]
        D = means.loc["D_ood_missing", metric]
        gap = D - B - C + A
        interaction_rows.append({"model": model_name, "metric": metric,
                                  "A": A, "B": B, "C": C, "D": D, "interaction_gap": gap})
        print(f"{model_name:10s} {metric:8s}  A={A:.4f} B={B:.4f} C={C:.4f} D={D:.4f}  gap={gap:+.4f}")

interaction_df = pd.DataFrame(interaction_rows)
interaction_df.to_csv("pilot_interaction_gaps.csv", index=False)

print("\nFiles written: pilot_results_raw.csv, pilot_results_summary.csv, pilot_interaction_gaps.csv")
print("Go/no-go rule of thumb: if interaction_gap is clearly positive and larger "
      "than the seed-to-seed std of D, you likely have a real compound effect "
      "worth scaling to the full 8-dataset / 3-mechanism design.")

=== Interaction check: gap = (D-A) - [(B-A)+(C-A)] = D - B - C + A ===
Positive gap = evidence of a non-additive (interactive) compound failure.

LightGBM   ECE       A=0.0162 B=0.0143 C=0.0419 D=0.0592  gap=+0.0191
LightGBM   Brier     A=0.2123 B=0.2218 C=0.2278 D=0.2361  gap=-0.0012
LightGBM   LogLoss   A=0.6122 B=0.6335 C=0.6455 D=0.6636  gap=-0.0031
TabPFN     ECE       A=0.0176 B=0.0272 C=0.0347 D=0.0407  gap=-0.0036
TabPFN     Brier     A=0.2090 B=0.2194 C=0.2220 D=0.2310  gap=-0.0014
TabPFN     LogLoss   A=0.6039 B=0.6278 C=0.6311 D=0.6513  gap=-0.0037

Files written: pilot_results_raw.csv, pilot_results_summary.csv, pilot_interaction_gaps.csv
Go/no-go rule of thumb: if interaction_gap is clearly positive and larger than the seed-to-seed std of D, you likely have a real compound effect worth scaling to the full 8-dataset / 3-mechanism design.


In [ ]:
#11


In [17]:
# Cell 10 (revised) — pairs each seed's B and D together and reports the
# gap's own mean and std directly, instead of computing gap from separately
# averaged B and D (which hides exactly the noise issue we just found)
print("=== Interaction check (per-seed paired gap) ===\n")

interaction_rows = []
for model_name in models:
    df_m = results_df[results_df.model == model_name]
    A = df_m[df_m.condition == "A_iid_complete"][["ECE","Brier","LogLoss"]].iloc[0]
    C = df_m[df_m.condition == "C_ood_complete"][["ECE","Brier","LogLoss"]].iloc[0]
    B_by_seed = df_m[df_m.condition == "B_iid_missing"].set_index("seed")[["ECE","Brier","LogLoss"]]
    D_by_seed = df_m[df_m.condition == "D_ood_missing"].set_index("seed")[["ECE","Brier","LogLoss"]]

    for metric in ["ECE", "Brier", "LogLoss"]:
        gaps = D_by_seed[metric] - B_by_seed[metric] - C[metric] + A[metric]
        mean_gap, std_gap = gaps.mean(), gaps.std()
        print(f"{model_name:10s} {metric:8s}  gap={mean_gap:+.4f}  std={std_gap:.4f}  "
              f"(n={len(gaps)} seeds)")
        interaction_rows.append({"model": model_name, "metric": metric,
                                  "gap_mean": mean_gap, "gap_std": std_gap})

pd.DataFrame(interaction_rows).to_csv("pilot_interaction_gaps.csv", index=False)

=== Interaction check (per-seed paired gap) ===

LightGBM   ECE       gap=+0.0191  std=0.0014  (n=5 seeds)
LightGBM   Brier     gap=-0.0012  std=0.0010  (n=5 seeds)
LightGBM   LogLoss   gap=-0.0031  std=0.0021  (n=5 seeds)
TabPFN     ECE       gap=-0.0036  std=0.0063  (n=5 seeds)
TabPFN     Brier     gap=-0.0014  std=0.0009  (n=5 seeds)
TabPFN     LogLoss   gap=-0.0037  std=0.0020  (n=5 seeds)
